# Lab 4: Parameter-efficient fine-tuning

Fine-tuning all parameters of pre-trained language models can be resource-intensive. Because of this, current research in natural language processing is looking into developing methods for adapting models to downstream tasks without full fine-tuning. These methods only tune a small number of model parameters while yielding performance comparable to that of a fully fine-tuned model.

OWN NOTES: fine-tuning: the process of ffurther training a pre-trained model on a smaller, specialized dataset to adapt it to a specific niche. 

In this lab, you will implement LoRA, one of the most well-known methods for parameter-efficient fine-tuning. LoRA stands for “Low-Rank Adaptation of Large Language Models” and was originally described in a research article by [Hu et al. (2021)](https://arxiv.org/abs/2106.09685).

Along the way, you will earn experience with [Hugging Face Transformers](https://huggingface.co/docs/transformers/en/index), a state-of-the-art library for training and deploying language models, as well as with several related libraries. In particular, you will learn a best-practice workflow for downloading a Transformer model and fine-tuning it on the downstream task of binary sentiment classification.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Dataset

The data for this lab comes from the [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/). The full dataset consists of 50,000 highly polar movie reviews collected from the Internet Movie Database (IMDB). Here, we use a random sample consisting of 2,000 reviews for training and 500 reviews for evaluation.

To load the dataset, we use the [Hugging Face Datasets](https://huggingface.co/docs/datasets/en/index) library.

In [1]:
from datasets import load_dataset

imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

imdb_dataset

/courses/TDDE09/tdde09/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label'],
        num_rows: 500
    })
})

As we can see, each sample in the dataset is a record with three fields: an internal index (`index`, an integer), the text of the review (`review`, a string), and the sentiment label (`label`, an integer – 1&nbsp;for “positive” and 0&nbsp;for “negative” sentiment).

Here is an example record:

In [2]:
imdb_dataset["train"][645]

{'index': 2981,
 'review': 'Brilliant execution in displaying once and for all, this time in the venue of politics, of how "good intentions do actually pave the road to hell". Excellent!',
 'label': 1}

## Tokeniser

As our pre-trained language model, we will use [DistilBERT](https://huggingface.co/docs/transformers/en/model_doc/distilbert), a compact encoder model with 40% less parameters than BERT base. DistilBERT is not actually a *large* language model by modern standards and thus does not benefit as much from parameter-efficient fine-tuning as other models. However, it has the benefit of being light and fast, and can be run even on consumer hardware.

To feed the movie reviews to DistilBERT, we need to tokenise them and encode the resulting tokens as integers in the model vocabulary. We start by loading the DistilBERT tokeniser using the [Auto classes](https://huggingface.co/docs/transformers/en/model_doc/auto):

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
#tokenizer(imdb_dataset["train"][645]["text"], truncation=True)

In [4]:
sample_text = "In Panic In The Streets"
tokenized_output = tokenizer(sample_text, truncation=True)
print(tokenized_output)

{'input_ids': [101, 1999, 6634, 1999, 1996, 4534, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


We then create a tokenised version of the dataset:

In [5]:
def tokenize_function(batch): #batch is a dictionary with keys "review" and "label" 
    return tokenizer(batch["review"], padding=True, truncation=True)


tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)

tokenized_imdb_dataset
print("ID LIST1:", tokenized_imdb_dataset["train"]["input_ids"][0])

ID LIST1: [101, 1999, 6634, 1999, 1996, 4534, 2957, 15536, 22117, 17007, 3248, 1057, 1012, 1055, 1012, 3212, 3460, 2040, 2038, 2010, 2733, 12726, 2135, 7153, 2007, 1037, 11547, 2008, 3397, 11629, 1012, 2004, 8872, 2703, 5203, 7919, 2685, 2041, 1996, 3124, 2351, 2013, 2048, 10432, 1999, 1996, 3108, 1012, 2008, 1005, 1055, 2025, 1996, 3277, 2182, 1010, 1996, 2048, 1997, 2068, 2468, 15175, 5826, 1999, 2019, 3947, 2000, 2424, 1996, 15978, 1998, 3087, 2842, 6086, 2000, 1996, 4295, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 2004, 2001, 4197, 2041, 2011, 2151, 2193, 1997, 2111, 1010, 2005, 2070, 3114, 2472, 12005, 2050, 22001, 2106, 2025, 8572, 2000, 3459, 1996, 2235, 3033, 2007, 3087, 2008, 4165, 2066, 2027, 1005, 2128, 2013, 5773, 1012, 2383, 2042, 2000, 2047, 5979, 2073, 1996, 2466, 3138, 2173, 1045, 2064, 7714, 2012, 22199, 2000, 2008, 1012, 2957, 15536, 22117, 17007, 1998, 2010, 2564, 6437, 19337, 16216, 14141, 2229, 2064, 2022, 28512, 2138, 2004, 1037, 3212, 3460, 2002, 2071,

In [6]:
batch_of_text = ["I loved it!", "This is a bit longer text that should be truncated."]
tokenized_batch = tokenizer(batch_of_text, padding=True, truncation=True)
print("ID LIST1:", tokenized_batch["input_ids"][0])
print("ID LIST2:", tokenized_batch["input_ids"][1]) 
print("ATTENTION MASK1:", tokenized_batch["attention_mask"][0])
print("ATTENTION MASK2:", tokenized_batch["attention_mask"][1])

ID LIST1: [101, 1045, 3866, 2009, 999, 102, 0, 0, 0, 0, 0, 0, 0]
ID LIST2: [101, 2023, 2003, 1037, 2978, 2936, 3793, 2008, 2323, 2022, 25449, 1012, 102]
ATTENTION MASK1: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
ATTENTION MASK2: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


As we can see, tokenising adds two additional fields to each review: `input_ids` is the list of token ids corresponding to the review, and `attention_mask` is the list of indices specifying which tokens the encoder should attend to.

To avoid trouble when fine-tuning the model later, the next cell disables tokeniser parallelism.

In [7]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Trainer

In this section, we will set up our workflow for training and evaluating DistilBERT models. The central component in this workflow is the [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer), which provides extensive configuration options. Here, we leave most of these options at their default value. Two changes we *do* make are to enable evaluation of the trained model after each epoch, and to log the training and evaluation loss after every 5&nbsp;training steps (the default is 500).

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments( # This class is to hold the hyperparmeters to controll the proccess of learning
    output_dir="tmp_trainer",       # the directory where tehe trainer checkpoints are saved
    eval_strategy="epoch",          # An epoch is one complete pass through the entire training datasset. 
    logging_steps=5,                # to print the error rate every 5 steps. 
)

In addition to the loss, we also track classification accuracy. For this we import the [Hugging Face Evaluate](https://huggingface.co/docs/evaluate/en/index) library and define a small helper function `compute_metrics()` that the trainer will call after each epoch.

In [9]:
import evaluate

accuracy = evaluate.load("accuracy") # this loads a predefined metric for calculating the accuracy of the model's predictions.


def compute_metrics(eval_pred): # eval_pred is a tuple of (logits, labels), where logits are the raw predictions from the model and labels are the true labels for the evaluation dataset.
    logits, labels = eval_pred # for example logits might be something like [[0.2, 0.8], [0.6, 0.4]], where each inner list represents the predicted probabilities for each class, and labels might be something like [1, 0], where each element represents the true label for the corresponding example in the evaluation dataset.
    predictions = logits.argmax(axis=-1) # for example the logits might be something like [[0.2, 0.8], [0.6, 0.4]], where each inner list represents the predicted probabilities for each class. The argmax function would then return [1, 0], indicating that the model predicts the second class for the first example and the first class for the second example.
    return accuracy.compute(predictions=predictions, references=labels) # this line computes the accuracy by comparing the predicted labels with the true labels. The compute method of the accuracy metric takes in the predicted labels and the true labels and returns a dictionary containing the computed accuracy value. 
                                                                #for example, if the predicted labels are [1, 0] and the true labels are [1, 0], the accuracy would be 1.0 (100% accuracy). If the predicted labels are [1, 0] and the true labels are [0, 1], the accuracy would be 0.0 (0% accuracy). If the predicted labels are [1, 0] and the true labels are [1, 1], the accuracy would be 0.5 (50% accuracy).

In the next cell we define a convenience function `make_trainer()` that creates a readily-configured trainer for a specified model (*model*). We will use this trainer both to train the model on the training section of the tokenised review dataset, and to evaluate it on the evaluation section.

In [10]:
from transformers import Trainer


def make_trainer(model): # this function takes a model as input and creates a Trainer object that will be used to train the model on the tokenized IMDB dataset. The Trainer is initialized with the model, training arguments, training dataset, evaluation dataset, and the compute_metrics function that will be used to evaluate the model's performance during training.
    trainer = Trainer(
        model=model, # the model to be trained
        args=training_args, # the training arguments that control the training process
        train_dataset=tokenized_imdb_dataset["train"], # the tokenized training dataset
        eval_dataset=tokenized_imdb_dataset["eval"], # the tokenized evaluation dataset
        compute_metrics=compute_metrics, # the function that computes the evaluation metrics during training
    )
    return trainer # the function returns the created Trainer object that can be used to train the model. for example, you can call this function with a pre-trained model like this: trainer = make_trainer(pretrained_model) and then use trainer.train() to start the training process.

## Full fine-tuning

In the rest of this notebook, we will work our way to the implementation of LoRA, and compare LoRA to traditional fine-tuning methods. Our first point of reference is a fully fine-tuned DistilBERT model.

We start by loading the pre-trained model:

In [11]:
from transformers import AutoModelForSequenceClassification

pretrained_model = AutoModelForSequenceClassification.from_pretrained( # this line loads a pre-trained model for sequence classification. The model is based on the "distilbert-base-uncased" architecture and is fine-tuned for a binary classification task (num_labels=2). This means that the model has been trained to classify input sequences into two classes, which is suitable for tasks like sentiment analysis where the goal is to classify text as positive or negative.
    "distilbert-base-uncased", num_labels=2
)

pretrained_model

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1520.28it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


The architecture of DistilBERT is that of a standard Transformer encoder with an embedding layer (`embeddings`) followed by a stack of six Transformer blocks (`transformer`) and a feedforward network with two linear layers (`pre_classifier` and `classifier`) and a final dropout layer (`dropout`).

### 🧩 Task 4.01: Counting the number of trainable parameters

One relevant measure in the context of parameter-efficient fine-tuning is the number of parameters that need to be changed when training a model. Your first task in this lab is to write a function `num_trainable_parameters()` that calculates this number for a given model.

In [12]:
def num_trainable_parameters(model):
    # TODO: Replace the next line with your own code
    return sum(p.numel() for p in model.parameters() if p.requires_grad) 
# model.parameters() returns an iterator over all the parameters of the model.
# p.numel() returns the total number of elements in the parameter tensor p. a for example if a weight matrix has shape (10, 10), then p.numel() would return 10 * 10 = 100.
# The if p.requires_grad: if true -> track the gradients for the tensor that the optimizer can update it during training (trainable).
# if false -> the tensor is frozen and will changers, so we don't count it as a trainable parameter.
print(f"Number of trainable parameters: {num_trainable_parameters(pretrained_model)}") 

# nn.Linear(3,5) -> with 3 as input features and 5 as output features, the number of parameters would be (3 * 5) + 5 = 20. for example in the case of movies reviews, if we have a movie with 3 features (e.g., acting, plot, cinematography) and we want to classify it into 5 genres (e.g., action, comedy, drama, horror, sci-fi).
# The gradient the mathematical compass that tells the network how to learn. for example when the model makes a predictio (e.g., guessing a review is positive when it's actually negative), the system calculates the error score (the LOSS) and then uses the gradient to adjust the model's parameters in a way that reduces the error in future predictions. The more trainable parameters a model has, the more complex patterns it can learn, but it also requires more data and computational resources to train effectively.


Number of trainable parameters: 66955010


The function should implement the following specification:

> **num_trainable_parameters** (*model*)
>
> Returns the number of float-valued trainable parameters in the specified *model* as an integer.

#### 👍 Hint

The term *parameter* can refer to either complete tensors or the individual elements of these tensors. For example, a linear layer created by `nn.Linear(3, 5)` has 2&nbsp;tensor-valued parameters (a weight matrix and a bias vector) and 20&nbsp;float-valued parameters (the elements of these tensors). To get the tensor-valued parameters of a model, you can use the [`parameters()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.parameters) method. A parameter is *trainable* if it requires gradient.

#### 🤞 Test your code

To test your code, apply your function to the pre-trained model. The correct number of float-valued trainable parameters for this model is 66,955,010.

### Fine-tuning

When we load the pre-trained model, the Hugging Face Transformers library warns us that the weights of the feedforward network have not yet been trained. To do so, in the next cell, we pass the pre-trained model to a trainer and initiate the fine-tuning process.

**⚠️ Please note that fine-tuning the model will take some time! ⚠️**

You can work on the other problems in this lab while you are waiting.

In [13]:
finetuned_trainer = make_trainer(pretrained_model)

finetuned_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.529806,0.261193,0.910000
2,0.032442,0.362609,0.910000
3,0.013909,0.394386,0.908000


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.91s/it]


TrainOutput(global_step=750, training_loss=0.22262191364293296, metrics={'train_runtime': 136.4407, 'train_samples_per_second': 43.975, 'train_steps_per_second': 5.497, 'total_flos': 794804391936000.0, 'train_loss': 0.22262191364293296, 'epoch': 3.0})

Because full fine-tuning is so resource-intensive, we save the fine-tuned model to disk:

In [14]:
finetuned_trainer.save_model("finetuned")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.84s/it]


Later in this notebook, whenever you need the fully fine-tuned version of the model, you can load it as follows:

In [15]:
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2033.50it/s, Materializing param=pre_classifier.weight]                                  


### Convenience functions

Because we will repeat the steps we just took to fine-tune the pre-trained model several times in this notebook, we define two convenience functions:

In [16]:
def train(model):
    print("Number of trainable parameters:", num_trainable_parameters(model))
    trainer = make_trainer(model)
    trainer.train()
    return model


In [17]:
def evaluate(model):
    trainer = make_trainer(model)
    return trainer.evaluate()

## Tuning the final layers only

If full fine-tuning marks one end of the complexity spectrum, the other end is marked by only tuning the final layers of the transformer – the *head* of the model. In the case of DistilBERT, the head consists of the `pre_classifier` and `classifier` layers.

### 🧩 Task 4.02: Head-tuning

Implement the head-tuning strategy by coding the following function:

In [18]:

from torch import mode


def make_headtuned_model():
    # TODO: Replace the next line with your own code


    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    for param in model.parameters():
        param.requires_grad = False # this line freezes all the parameters of the model, meaning that they will not be updated during training. This is done to prevent the pre-trained weights from being modified, allowing us to only train the newly added classification head.
    for param in model.pre_classifier.parameters():
        param.requires_grad = True  # this line unfreezes the parameters of the pre-classifier layer, allowing them to be updated during training. The pre-classifier layer is a part of the model that is responsible for processing the output from the transformer layers before it is passed to the classification head. By unfreezing these parameters, we allow the model to fine-tune the pre-classifier layer to better adapt to the specific task at hand, while still keeping the rest of the model frozen to preserve the pre-trained knowledge.
    for param in model.classifier.parameters():
        param.requires_grad = True # this line unfreezes the parameters of the classification head, allowing them to be updated during training. The classification head is the final layer of the model that takes the output from the pre-classifier layer and produces the final predictions for the classification task. By unfreezing these parameters, we allow the model to fine-tune the classification head to better adapt to the specific task at hand, while still keeping the rest of the model frozen to preserve the pre-trained knowledge.
    return model
    #raise NotImplementedError



In [19]:
head_params = num_trainable_parameters(make_headtuned_model())
print(f"Number of trainable parameters in head-tuned model: {head_params}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2949.33it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters in head-tuned model: 592130


Here is the specification of this function:

> **make_headtuned_model** ()
>
> Returns a model that is identical to the pre-trained model, except that the head layers have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.)

#### 👍 Hint

You freeze a parameter by setting its `requires_grad`-attribute to `False`.

Once you have an implementation of the head-tuning strategy, evaluate it on the evaluation data. How much accuracy do we lose when only training the final layers of the pre-trained model, compared to full fine-tuning?

In [20]:
headtuned_model = make_headtuned_model()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1889.59it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### 🤞 Test your code

If you configured your model correctly, `num_trainable_parameters()` should show 592,130 trainable parameters.

For future reference, we also save the head-tuned model:

In [21]:
make_trainer(headtuned_model).save_model("headtuned")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.36s/it]


## Layer surgery

LoRA works by “wrapping” frozen layers from the pre-trained Transformer model inside adapter modules. Conventionally, this wrapping is only applied to the linear layers that transform the queries and values in the self-attention mechanism. To implement the wrapping, we need functions to extract and replace layers in a model. Your task in this section is to code these functions.

### 🎓 Task 4.03: Extracting layers

Code a function that extracts the query and value linear layers from a DistilBERT model:

In [22]:
def extract(model):
    # TODO: Replace the next line with your own code

    extracted_layers = {}

    for i in range(6): # this loop iterates over the 6 transformer layers of the DistilBERT model. 

        q_name = f"distilbert.transformer.layer.{i}.attention.q_lin" # this line constructs the name of the query linear layer for the i-th transformer layer. The q_lin layer is responsible for transforming the input into query vectors that are used in the attention mechanism of the transformer. for example, in the first transformer layer (i=0), the q_name would be "distilbert.transformer.layer.0.attention.q_lin", which corresponds to the query linear layer of the first transformer layer. In the second transformer layer (i=1), the q_name would be "distilbert.transformer.layer.1.attention.q_lin", and so on for all 6 layers.
        v_name = f"distilbert.transformer.layer.{i}.attention.v_lin" # this line constructs the name of the value linear layer for the i-th transformer layer. The v_lin layer is responsible for transforming the input into value vectors that are used in the attention mechanism of the transformer.

        extracted_layers[q_name] = model.get_submodule(q_name) # this line retrieves the query linear layer from the model using the get_submodule method and stores it in the extracted_layers dictionary with the key being the name of the layer (q_name). The get_submodule method allows us to access specific layers of the model by their names, which is useful for extracting and analyzing specific components of the model.
        extracted_layers[v_name] = model.get_submodule(v_name)

    return extracted_layers

In [23]:
extracted_layers = extract(pretrained_model)
print(extracted_layers)

{'distilbert.transformer.layer.0.attention.q_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.0.attention.v_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.1.attention.q_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.1.attention.v_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.2.attention.q_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.2.attention.v_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.3.attention.q_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.3.attention.v_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.4.attention.q_lin': Linear(in_features=768, out_features=768, bias=True), 'distilbert.transformer.layer.4.attention.v_lin': Linear(in_fea

Implement this function to match the following specification:

> **extract** (*model*)
>
> Takes a DistilBERT model (*model*) and extracts the query and value linear layers from each block of the Transformer. Returns a dictionary mapping the DistilBERT module names of these layers to the layers themselves (instances of `nn.Linear`).

#### 👍 Hint

As we saw earlier, the DistilBERT model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. Use [`get_submodule()`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.get_submodule) to retrieve a layer by name. You can hard-wire the names of the layers you want to extract.

#### 🤞 Test your code

To test your code, check the number of trainable float-valued parameters in the extracted layers. This number should be 7,087,104.

In [24]:
total_tensors = 0
total_floats = 0

for name, layer in extracted_layers.items():
    for pram in layer.parameters():
        if pram.requires_grad:
            total_tensors += 1
            total_floats += pram.numel()

print (f"total tensors: {total_tensors}, total float values: {total_floats}")

total tensors: 24, total float values: 7087104


### 🎓 Task 4.04: Replacing layers

Next, code the inverse of the `extract()` function to replace selected layers of a module using a dictionary of named layers.

In [25]:
def replace(model, named_layers): # this function takes a model and a dictionary of named layers as input and replaces the specified layers in the model with the new layers provided in the dictionary. The named_layers dictionary should have keys that correspond to the names of the layers to be replaced in the model, and values that are the new layer objects that will replace the existing layers in the model. The function iterates through the named_layers dictionary, retrieves the corresponding layer from the model using the get_submodule method, and replaces it with the new layer using setattr. Finally, the modified model is returned.
    # TODO: Replace the next line with your own code

    for name, new_layer in named_layers.items():
        parts = name.split(".")
        current_module = model
        for part in parts[:-1]:
            current_module = getattr(current_module, part)
        setattr(current_module, parts[-1], new_layer)
    return model

Implement this function to match the following specification:

> **replace** (*model*, *named_layers*)
>
> Takes a DistilBERT model (*model*) and a dictionary in the format returned by `extract()` (*named_layers*) and injects the extracted layers into the model. More specifically, suppose that *named_layers* contains a key–value pair `(name, layer)`. Then the function replaces the submodule of *model* addressed by the fully-qualified string name `name` by the layer `layer`. Returns the modified model.

#### 👍 Hint

Use [`getattr()`](https://docs.python.org/3/library/functions.html#getattr) and [`setattr()`](https://docs.python.org/3/library/functions.html#setattr) to return or set the value of a named submodule.

#### 🤞 Test your code

To test your implementation, write code that (1)&nbsp;extracts the query and value linear layers from the fine-tuned model; (2)&nbsp;replaces these layers with clones with random weights; and (3)&nbsp;replaces these layers again with the original versions. Evaluating the modified model after step&nbsp;(2) should yield a near-random accuracy. Evaluating it again after step&nbsp;(3) should yield the original accuracy.

The following function should be helpful. It clones a linear layer, copying the weights and the bias from the original.

In [26]:
import torch.nn as nn


def clone_linear(original):
    out_features, in_features = original.weight.shape
    copy = nn.Linear(in_features, out_features)
    copy.load_state_dict(original.state_dict())
    return copy

## Low-rank approximation

The basic idea behind LoRA is to conceptualise fine-tuned weights as a sum $W_0 + \Delta W$ of the weights from the pre-trained model, $W_0$, and a low-rank update matrix $\Delta W$. The goal of fine-tuning, then, is to learn the update matrix; this happens in the adapter layers.

Before we get to the implementation of the LoRA adapter layers, we first check to what extent the assumption that fine-tuning can be described by low-rank matrices holds true for DistilBERT. To do so, we will “cheat” and replace the query and value linear layers of the head-tuned model with low-rank approximations. The technical key to this is the truncated singular value decomposition (SVD).

### 🎓 Task 4.05: Low-rank matrix approximation

Your first task in this section is to implement the low-rank matrix approximation.

In [27]:
import torch


def approximate(matrix, rank): # matrix has the shape (768, 384) and rank is the desired rank for the approximation. The function performs a low-rank approximation of the input matrix using singular value decomposition (SVD). The SVD decomposes the matrix into three matrices U, S, and V such that matrix = U * S * V^T. By truncating U, S, and V to the specified rank, we can create a low-rank approximation of the original matrix that captures the most important features while reducing the dimensionality.
    # TODO: Replace the next line with your own code
    # perform the singular value decomposition (SVD) of the matrix, which decomposes it into three matrices U, S, and V such that matrix = U * S * V^T. The full_matrices=False argument ensures that we get the reduced SVD.
    # full_matrices=False means that we want to compute the reduced SVD, which returns only the non-zero singular values and their corresponding singular vectors. This is more efficient than computing the full SVD when the matrix has low rank.
    u, s, v = torch.linalg.svd(matrix, full_matrices=False)

    u_truncated = u[:, :rank] # the shape of u is (768, 384), so u[:, :rank] will give us the first rank columns of u, resulting in a matrix of shape (768, rank). T[768, 384] -> (768, 8)
    s_truncated = s[:rank] # the shape of s is (384,), so s[:rank] will give us the first rank singular values, resulting in a vector of shape (rank,). T[384] -> (8,)
    v_truncated = v[:rank, :] # the shape of v is (384, 384), so v[:rank, :] will give us the first rank rows of v, resulting in a matrix of shape (rank, 384). T[384, 384] -> (8, 384)

    approximation = u_truncated @ torch.diag(s_truncated) @ v_truncated # this formula is M=U * S * V^T, where U is the matrix of left singular vectors, S is the diagonal matrix of singular values, and V^T is the transpose of the matrix of right singular vectors. By truncating U, S, and V to the specified rank, we can create a low-rank approximation of the original matrix that captures the most important features while reducing the dimensionality. 
    return approximation # the final shape of the approximation will be (768, 384), but it will have a lower rank than the original matrix, meaning that it will capture the most important features while reducing the dimensionality. The approximation can be used to replace the original weight matrix in the model, allowing for a more efficient representation while still maintaining good performance on the task at hand.

Implement this function to match the following specification:

> **approximate** (*matrix*, *rank*)
>
> Takes a 2D-tensor (*matrix*) and an integer rank $r$ (*rank*), computes the truncated SVD with rank $r$ on the tensor, and returns the corresponding low-rank approximation matrix.

#### 👍 Hint

If you need a refresher on the low-rank matrix approximation, read the corresponding section from the Wikipedia article on the [Singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition#Low-rank_matrix_approximation). The truncated SVD is an extension of the full SVD; the latter can be computed using [`torch.linalg.svd()`](https://pytorch.org/docs/stable/generated/torch.linalg.svd.html).

#### 🤞 Test your code

To test your code, run the following cell. It creates a matrix `original` with rank $r \leq 8$ and after that the rank-$8$ approximation matrix `approximation`. You should find that the distance between the two matrices is very low.

In [28]:
original = torch.rand(768, 8) @ torch.rand(8, 384)
approximation = approximate(original, 8)
torch.dist(original, approximation)

print("Original matrix shape:", original.shape)
print("Approximation matrix shape:", approximation.shape)   


Original matrix shape: torch.Size([768, 384])
Approximation matrix shape: torch.Size([768, 384])


### 🎓 Task 4.06: Approximated fine-tuned model (version 1)

In the next step, your task is to construct a version of the head-tuned model in which every query and value linear layer is replaced by a low-rank approximation of the corresponding layer from the fully fine-tuned model.

In [29]:
def make_approximated_model_1(rank): 
    # TODO: Replace the next line with your own code

    # load the two reference models
    headtuned_model = make_headtuned_model()
    finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

    #extract the q_lin and v_lin layers from the headtuned model
    finetuned_layers = extract(finetuned_model)

    #prepare the dictionary of approximated layers
    approximated_layers = {}

    #loop over the extraced layesr and compute their approximations
    for name, layer in finetuned_layers.items(): #for each layer in the finetuned model, we will compute a low-rank approximation of its weight matrix and create a new linear layer with the same input and output dimensions as the original layer. We will then replace the original layer in the headtuned model with the new approximated layer.
        weight_matrix = layer.weight.data # this line retrieves the weight matrix of the current layer and stores it in the variable weight_matrix. 
        bias_vector = layer.bias.data if layer.bias is not None else None

        compressed_weight_matrix = approximate(weight_matrix, rank) # this line calls the approximate function to compute a low-rank approximation of the weight matrix using the specified rank. The resulting compressed_weight_matrix will have the same shape as the original weight matrix but with a lower rank, meaning that it will capture the most important features while reducing the dimensionality.

        out_features, in_features = compressed_weight_matrix.shape 
        new_layer = nn.Linear(in_features, out_features, bias=(bias_vector is not None))

        new_layer.weight.data = compressed_weight_matrix
        if bias_vector is not None:
            new_layer.bias.data = bias_vector

        approximated_layers[name] = new_layer
    return replace(headtuned_model, approximated_layers)

Here is the specification of this function:

> **make_approximated_model_1** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which every query and value linear layer is replaced by its $r$-approximated corresponding layer from the fully fine-tuned model.

Run the next cell to evaluate your model for different rank values. Start with the full rank and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model?

ANSWER: Rank=768 -> accuracy=0.476       Rank=384 -> accuraccy=0.526        Rank=320 -> accuracy= 0.456            So the lowest rank that gives a higher accuracy than the head-tunned model is 384. 

In [30]:
approximated_model_1 = make_approximated_model_1(384)

evaluate(approximated_model_1)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1965.47it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2496.61it/s, Materializing param=

{'eval_loss': 0.6960251331329346,
 'eval_model_preparation_time': 0.0005,
 'eval_accuracy': 0.474,
 'eval_runtime': 2.7295,
 'eval_samples_per_second': 183.183,
 'eval_steps_per_second': 23.081}

### 🎓 Task 4.07: Approximated fine-tuned model (version 2)

In the approximated model from the previous section, the truncated SVD is applied to the full weight matrix of the fine-tuned model: $W_0 + \Delta W$. In LoRA, the low-rank approximation only applies to the *update matrix* $\Delta W$, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

In [41]:
def make_approximated_model_2(rank):
    # TODO: Replace the next line with your own code

    # load the  baseline model
    headtuned_model = make_headtuned_model()
    finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

    baseline_layers = extract(headtuned_model)
    finetuned_layers = extract(finetuned_model)

    approximated_layers = {}

    for name in baseline_layers.keys():

        w_0 = baseline_layers[name].weight.data
        w_tunned = finetuned_layers[name].weight.data

        delta_w = w_tunned - w_0
        delta_w_approx = approximate(delta_w, rank)

        final_weight = w_0 + delta_w_approx

        bias_vector = finetuned_layers[name].bias.data if finetuned_layers[name].bias is not None else None

        out_features, in_features = final_weight.shape
        new_layer = nn.Linear(in_features, out_features, bias=(bias_vector is not None))
        new_layer.weight.data = final_weight
        if bias_vector is not None:
            new_layer.bias.data = bias_vector

        approximated_layers[name] = new_layer

    return replace(headtuned_model, approximated_layers)

Implement the function to match the following specification:

> **make_approximated_model_2** (*rank*)
>
> Takes an integer rank $r$ (*rank*) and returns a version of the head-tuned model in which the weight matrix of every query and value linear layer is replaced by the sum $W_0 + \Delta W$, where $W_0$ is the weight matrix of the pre-trained model and $\Delta W$ is the rank-$r$ approximation of the update matrix, i.e., the difference between the fully fine-tuned weights and the pre-trained weights.

Run the next cell to evaluate your model for different rank values. Start with the rank from the approximated model from the previous section and then halve the rank in each step. What is the lowest rank that still gives you a higher accuracy than the head-tuned model? 

ANSWER:  the lowest rank is = 3.

In [32]:
approximated_model_2 = make_approximated_model_2(768)

evaluate(approximated_model_2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3376.65it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3260.41it/s, Materializing param=

{'eval_loss': 0.6962202787399292,
 'eval_model_preparation_time': 0.0005,
 'eval_accuracy': 0.446,
 'eval_runtime': 2.7551,
 'eval_samples_per_second': 181.478,
 'eval_steps_per_second': 22.866}

In [33]:
approximated_model_2 = make_approximated_model_2(3)

evaluate(approximated_model_2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2628.92it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2766.08it/s, Materializing param=

{'eval_loss': 0.6951589584350586,
 'eval_model_preparation_time': 0.0005,
 'eval_accuracy': 0.474,
 'eval_runtime': 2.7081,
 'eval_samples_per_second': 184.631,
 'eval_steps_per_second': 23.263}

## Low-Rank Adaptation (LoRA)

In this section, you will implement the LoRA adapters and fine-tune the adapted model.

### 🎓 Task 4.08: Implement the adapter

A LoRA adapter implements the forward function

$$
y = x W_0 + x \Delta W = x W_0 + x A B
$$

where $W_0$ is a linear transformation from the pre-trained model and $\Delta W$ is a learned update matrix, deconstructed into the product $AB$ of two rank-$r$ matrices $A$ and $B$. LoRA scales the update matrix $\Delta W$ by a factor of $\alpha / r$, where $\alpha$ is a hyperparameter. (To keep the formula tidy, we ignore the fact that the linear transformation in the pre-trained model may additionally include a bias.)



Own Note:

x: The input data (e.g., the numerical representation of a word).

W0​: The massive, original, frozen weight matrix from DistilBERT (e.g., shape 768×768).

xW0​: The output of the main pipe.

ΔW: The theoretical "update" matrix. This is what we would have learned if we did Full Fine-Tuning.

A and B: The two tiny matrices that make up our bypass pipe. Instead of learning a massive 768×768 ΔW, we learn A (shape 768×r) and B (shape r×768). When multiplied together (AB), they approximate ΔW.

y: The final combined output.

In [ ]:
import torch.nn as nn


class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()
        # TODO: Add your code here

        self.pretrained = pretrained # the original pre-trained model that we want to fine-tune using LoRA. This model will be used as the base for our LoRA implementation, and we will add low-rank adaptation layers on top of it to allow for efficient fine-tuning while keeping the original model's parameters frozen.
        self.rank = rank # the rank of the low-rank adaptation layers. This parameter controls the complexity of the adaptation and the number of trainable parameters in the LoRA model. A higher rank allows for a more expressive adaptation but also increases the number of trainable parameters, while a lower rank reduces the number of trainable parameters but may limit the expressiveness of the adaptation.
        self.alpha = alpha # the scaling factor for the LoRA adaptation. This parameter controls the magnitude of the updates from the low-rank adaptation layers during training. A higher alpha allows for larger updates, which can lead to faster convergence but may also cause instability during training, while a lower alpha results in smaller updates, which can lead to more stable training but may require more epochs to converge.

        in_features = pretrained.in_features # the number of input features for the linear layer that we want to adapt using LoRA. This is typically the same as the input dimension of the original linear layer in the pre-trained model that we are fine-tuning. For example, if we are fine-tuning a transformer model like DistilBERT, the in_features would be 768, which is the hidden size of the transformer layers.
        out_features = pretrained.out_features # the number of output features for the linear layer that we want to adapt using LoRA. This is typically the same as the output dimension of the original linear layer in the pre-trained model that we are fine-tuning. For example, if we are fine-tuning a transformer model like DistilBERT, the out_features would be 768, which is the hidden size of the transformer layers.

        self.lora_A = nn.Parameter(torch.randn(in_features, rank)) #the shape of lora_A will be (in_features, rank), which means that it will have the same number of rows as the input features of the original linear layer and a number of columns equal to the specified rank. This allows lora_A to learn a low-rank representation of the input features, which can capture the most important information while reducing the dimensionality. For example, if in_features is 768 and rank is 12, then lora_A will have a shape of (768, 12), allowing it to learn a low-rank representation of the input features that can be used for efficient fine-tuning.
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features)) # the shape of lora_B will be (rank, out_features), which means that it will have a number of rows equal to the specified rank and the same number of columns as the output features of the original linear layer. This allows lora_B to learn a low-rank representation of the output features, which can capture the most important information while reducing the dimensionality. For example, if rank is 12 and out_features is 768, then lora_B will have a shape of (12, 768), allowing it to learn a low-rank representation of the output features that can be used for efficient fine-tuning.

    def forward(self, x): # the input x is the input tensor that will be passed through the LoRA model.
        # TODO: Replace the next line with your own code

        output = self.pretrained(x)

        lora_output = (x @ self.lora_A @ self.lora_B)

        scaled_lora_output = lora_output * (self.alpha / self.rank)

        return output + scaled_lora_output

Your code must comply with the following specification:

**__init__** (*self*, *pretrained*, *rank* = 12, *alpha* = 24)

> Initialises the LoRA adapter. This sets up the matrices $A$ and $B$ from the equation above. The matrix $A$ is initialised with random weights from a standard normal distribution; the matrix $B$ is initialised with zeros. The argument *pretrained* is the linear layer from the pre-trained model that should be adapted. The arguments *rank* and *alpha* are the rank $r$ and the hyperparameter $\alpha$ in the equation above.

**forward** (*self*, *x*)

> Sends an input *x* through the adapter, implementing the equation above.

### 🎓 Task 4.09: Inject the adapter into the pre-trained model

The final step is to construct an adapted model by injecting the LoRA adapters into the pre-trained model.

In [42]:
def make_lora_model(rank):
    # TODO: Replace the next line with your own code

    model = make_headtuned_model()

    target_layers = extract(model)

    lora_layers = {}

    for name, layer in target_layers.items():

        wrapped_layer = LoRA(pretrained=layer, rank=rank, alpha=2 * rank)
        lora_layers[name] = wrapped_layer
        model = replace(model, lora_layers)

    return train(model)

Implement the function to match the following specification:

> **make_lora_model** (*rank*)
>
> Returns a model that is identical to the pre-trained model, except that the query and value linear layers have been wrapped in LoRA adapters, and the LoRA adapters and the head layers of the pre-trained model have been trained on the sentiment data. (The other parameters of the pre-trained model are left untouched.) The rank of the adapters is specified by the argument *rank*. The *alpha* value of the adapters is set to twice the rank (a common rule of thumb).

Run the next cell to evaluate your model for $r = 6$ and $\alpha = 12$. How many trainable parameters does the adapted model have? What accuracy do you get? How do these value relate to the number of trainable parameters and accuracy of the fully fine-tuned model, in terms of percentages?

ANSWER: The LoRA model trains roughly 702,722 parameters, which is ~1 % of the full model, we get a accuracy of ~89% (which very close to the full mode ~90%). 

In [43]:
lora_model = make_lora_model(6)


evaluate(lora_model)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3042.66it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Number of trainable parameters: 702722


Epoch,Training Loss,Validation Loss,Accuracy
1,0.525354,0.332144,0.870000
2,0.088069,0.338600,0.870000
3,0.095373,0.363975,0.890000


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.05s/it]


{'eval_loss': 0.36397525668144226,
 'eval_model_preparation_time': 0.0013,
 'eval_accuracy': 0.89,
 'eval_runtime': 2.8613,
 'eval_samples_per_second': 174.748,
 'eval_steps_per_second': 22.018}

## Alternatives to Transformer-based models

Even with methods for parameter-efficient fine-tuning, applying DistilBERT and other Transformer-based models comes at a significant cost – an investment that does not always pay off. In the final task of this lab, we ask you to explore a more traditional approach to classification and contrast it with the pre-training/fine-tuning approach of neural language models.

### 🎓 Task 4.10: Comparing with a non-neural classifier

Browse the web to find a tutorial on how to apply a classifier from the [scikit-learn](https://scikit-learn.org/stable/) library to the problem of sentiment classification and implement the method here in this notebook. We suggest you use [multinomial Naive Bayes](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) or [logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). (Once you have code for one method, it is easy to switch to the other.) Evaluate your chosen classifier on the IMDB dataset.

Questions to consider:

* Which classifier did you try? What results did you get? How long did it take you to train and run the classifier?

ANSWER: Classifier-> LogisticRegression,   Time->  0.02 seconds,  Results: LinearRegression accuracy-> ~84.40%   wile DistiBERT-> ~90%.
* What is your perspective on the trade-off between accuracy and resource requirements between the two approaches?

The Resource Penalty (DistilBERT): Neural networks require immense compute (GPUs), memory (to hold ~67 million parameters), and time.

The Accuracy Penalty (Logistic Regression): Classical ML models treat language as isolated keywords (Bag-of-Words). They cannot understand word order, sarcasm, or context. Therefore, you trade away the top 5-10% of accuracy.


* What did you learn? How, exactly, did you learn it? Why does this learning matter?

1: I learned it by physically extracting the Query and Value layers from the Transformer, applying Singular Value Decomposition (SVD) to prove the weights could be compressed into a lower rank, and writing a custom PyTorch adapter to inject a bypass mechanism into the model's forward pass

2: This matters because modern Large Language Models (like Llama-3 or GPT-4) are too massive to fine-tune conventionally. Understanding LoRA allows an engineer to take a generic, open-source 70-billion parameter model and specialize it for a specific company's needs using a single consumer GPU, making advanced AI accessible and cost-effective

In [44]:
train_texts = imdb_dataset["train"]["review"]
train_labels = imdb_dataset["train"]["label"]

eval_texts = imdb_dataset["eval"]["review"]
eval_labels = imdb_dataset["eval"]["label"]


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer # tTF: counts the number of times each word appears in a review
# IDF: measures how important a word is in the entire dataset by looking at how many reviews contain that word. The more reviews contain the word, the less important it is. TF-IDF combines these two measures to give a score for each word in each review, which can be used as features for machine learning models. For example, if the word "excellent" appears frequently in positive reviews but rarely in negative reviews, it will have a high TF-IDF score and can be a strong indicator of positive sentiment in the reviews.

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english') # this line initializes a TfidfVectorizer object with a maximum of 5000 features. The max_features parameter limits the number of features (i.e., unique words) that the vectorizer will consider when transforming the text data into numerical features. By setting max_features to 5000, we are telling the vectorizer to only keep the top 5000 most important features based on their TF-IDF scores, which can help reduce the dimensionality of the feature space and improve the performance of machine learning models trained on this data. For example, if we have a large dataset of movie reviews, setting max_features to 5000 will allow us to focus on the most relevant words in the reviews while ignoring less important ones, which can lead to better model performance and faster training times.
X_train = vectorizer.fit_transform(train_texts) # this line fits the TfidfVectorizer to the training texts and transforms them into a sparse matrix of TF-IDF features. The fit_transform method first learns the vocabulary from the training texts and then transforms the texts into a numerical representation based on the TF-IDF scores of the words in the reviews. The resulting X_train will be a sparse matrix where each row corresponds to a review and each column corresponds to a feature (i.e., a unique word), with the values representing the TF-IDF scores for each word in each review.
X_eval = vectorizer.transform(eval_texts) # this line transforms the evaluation texts into a


In [48]:
from sklearn.linear_model import LogisticRegression
import time

start_time = time.time()
logistic_model = LogisticRegression(max_iter=1000) # this line initializes a LogisticRegression
logistic_model.fit(X_train, train_labels) # this line fits the logistic regression model to the training data. The fit method takes in the feature matrix X_train and the corresponding labels train_labels and learns the parameters of the logistic regression model that best fit the training data. By setting max_iter to 1000, we are allowing the optimization algorithm to run for a maximum of 1000 iterations to find the optimal parameters for the model. This can help ensure that the model converges to a good solution, especially when dealing with larger datasets or more complex feature spaces.

end_time = time.time()
training_time = end_time - start_time
print(f"Training time for Logistic Regression: {training_time:.2f} seconds")

Training time for Logistic Regression: 0.02 seconds


In [53]:
from sklearn.metrics import accuracy_score

logistic_predictions = logistic_model.predict(X_eval) # this line uses the trained logistic regression model to make predictions on the evaluation data. The predict method takes in the feature matrix X_eval and returns the predicted labels for each review in the evaluation set based on the learned parameters of the logistic regression model.
logistic_accuracy = accuracy_score(eval_labels, logistic_predictions) # this line calculates the accuracy of the logistic regression model's predictions on the evaluation set. The accuracyScore function from sklearn.metrics

print(f"Logistic Regression Accuracy: {logistic_accuracy* 100:.2f}% ")

Logistic Regression Accuracy: 84.40% 


**🥳 Congratulations on finishing this lab! 🥳**